# Held-out result KV on new families, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

New `plan4_hold` families (seed 2070), 90 variants, disjoint from plan4_dev and plan4_fresh. Same five frozen conditions:

- `kv`: full completed cache.
- `kv_result`: RESULT-line KV only.
- `kv_sink`: the first four original cache positions only.
- `kv_result_sink`: RESULT-line KV plus those four positions.
- `text_result`: those RESULT lines as text.

A writes calculations once. B's instructions stay frozen. Capture decode-loop token ids. Report parseability, eligibility (including ineligible marked eligible), final answers, plan/answer disagreement, payload, process time, and family-clustered paired differences.

Do **not** load `plan4_dev` or `plan4_fresh`. Do not resume `results_stage_plan4_stable`. No budgets 32/64 or OBF.

Development reference: full KV 82/90 (eligibility 90/90), result KV 0/90, four-only 23/90, result+four 42/90 (eligibility 25/90), result text 79/90 (eligibility 90/90). After this run, stop expanding the pilot.

Download `/kaggle/working/results_stage_plan4_hold`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_hold.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_hold.py").is_file() and (cand / "data" / "plan4_hold.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_hold.py and data/plan4_hold.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_hold.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_hold", sum(1 for _ in (REPO / "data" / "plan4_hold.jsonl").open(encoding="utf-8")))

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_hold")
print("90 examples. Full KV, result KV, four-only, result+four, result text.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_hold", "--split", "plan4_hold"])
print("plan4 hold exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_hold"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_hold" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    print(path.read_text(encoding="utf-8")[:20000])